# Assignment: Hyperparameter Optimization with PyTorch & Optuna

## Learning Goals

By the end of this assignment, you should be able to:

- Build a configurable CNN in PyTorch
- Define and explore a hyperparameter search space
- Use Optuna to optimize model performance
- Properly separate training, validation, and test data

## Setup

In [3]:
!pip install optuna
!pip install torchmetrics
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchmetrics.classification import MulticlassAccuracy
from torchvision import datasets, transforms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 39.2 MB/s eta 0:00:00


In [4]:
IMAGE_WIDTH = 15
IMAGE_HEIGHT = IMAGE_WIDTH

NUMBER_OF_CHANNELS = 3

BATCH_SIZE = 64

NUMBER_OF_CLASSES = 10

RANDOM_SEED = 42

# Fix the seed for better reproducibility
torch.manual_seed(seed=RANDOM_SEED)
optuna_sampler = optuna.samplers.RandomSampler(seed=42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

## Data Loading

In [5]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:02<00:00, 76.3MB/s]


## Task 1: Create Train / Validation Split

We will optimize on validation data and only evaluate on the test set once.

Split the training dataset into:

- 80% training
- 20% validation

Use the [random_split](https://docs.pytorch.org/docs/2.11/data.html#torch.utils.data.random_split) function for this.

In [7]:
# TODO: Split dataset into 80% train and 20% validation
generator1 = torch.Generator().manual_seed(42)
train_size = 0.8 * len(dataset)
validation_size = len(dataset) - train_size

train_dataset, validation_dataset = random_split(dataset, [int(train_size) , int(validation_size)] , generator=generator1)

## Sanity Check

Before doing hyperparameter optimization, make sure your model can learn at all.

Run the cell below to:

- Train a small, fixed CNN for 3 epochs
- Confirm that loss decreases

In [8]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

number_of_filters = 16
kernel_size = 3

model = nn.Sequential(
    nn.Conv2d(NUMBER_OF_CHANNELS, number_of_filters, kernel_size),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(number_of_filters * IMAGE_WIDTH * IMAGE_HEIGHT, NUMBER_OF_CLASSES)
)

model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        predicted_values = model(x)
        loss = criterion(predicted_values, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch}: Loss = {total_loss:.4f}")

Epoch 0: Loss = 915.3745
Epoch 1: Loss = 756.9005
Epoch 2: Loss = 694.1141


## Task 2: Build a Dynamic CNN

You will implement a CNN with the following configurable parameters:

- Number of convolutional layers
- Number of filters
- Fully-connected layer size
- Dropout probability

Here, each conv block must follow:

`Conv2d → BatchNorm2d → ReLU → MaxPool2d`

In [16]:
class DynamicCNN(nn.Module):
    def __init__(
        self,
        number_of_conv_layers,
        number_of_filters,
        fully_connected_size,
        dropout_probability,
        # NOTE: kernel size is not varied here, because the images are too small for larger kernels with many max-pooling layers
        kernel_size: int = 3,
    ):
        super().__init__()

        self.convs = nn.ModuleList()
        in_channels = 3

        # Define padding for 'same' convolution to maintain spatial dimensions before pooling
        padding = kernel_size // 2

        for _ in range(number_of_conv_layers):
            # The output channels for each convolutional block will be `number_of_filters`
            block = nn.Sequential(
                # Add Conv2d layer
                nn.Conv2d(in_channels, number_of_filters, kernel_size, padding=padding),
                # Add BatchNorm2d
                nn.BatchNorm2d(number_of_filters),
                # Add ReLU
                nn.ReLU(),
                # Add MaxPool2d with a fixed kernel size of 2
                nn.MaxPool2d(kernel_size=2)
            )

            self.convs.append(block)

            # Update the input channels for the next block
            in_channels = number_of_filters

        self.dropout = nn.Dropout(p=dropout_probability)
        self.flattened_size = None

        self.fully_connected_1 = None
        self.fully_connected_2 = None
        self.fully_connected_size = fully_connected_size

    def forward(self, x):
        for block in self.convs:
            x = block(x)

        x = torch.flatten(x, 1)

        if self.flattened_size is None:
            self.flattened_size = x.size(1)
            self.fully_connected_1 = nn.Linear(
                in_features=self.flattened_size,
                out_features=self.fully_connected_size,
            )
            self.fully_connected_2 = nn.Linear(
                in_features=self.fully_connected_size,
                out_features=NUMBER_OF_CLASSES,
            )

        x = self.fully_connected_1(x)
        x = nn.functional.relu(x)
        x = self.dropout(x)
        x = self.fully_connected_2(x)

        return x

## Task 3: Define the Search Space

We will use Optuna to explore hyperparameters.

First, create a function to define the search space to traverse.

Choose the following values for the different hyperparameters:

- Number of convolutional layers: 1, 2, or 3
- Number of filters: 16, 32, or 64
- Fully-connected size: 64, 128, or 256
- Dropout probability: between 20% and 50%
- Learning rate: between 0.0001 and 0.01

Use the `suggest_*` methods of Optuna's [Trial class](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html) to define the values to choose from.

Hints:

- Use `suggest_int` for integer ranges
- Use `suggest_float` for float ranges
- Use `suggest_categorical` for discrete options
- Use **log scale** for the learning rate

In [17]:
def define_search_space(trial):
    parameters = {}

    # TODO: Define the search space
    parameters["number_of_conv_layers"] = 3
    parameters["number_of_filters"] = 64
    parameters["fully_connected_size"] = 256
    parameters["dropout_probability"] = 0.5
    parameters["learning_rate"] = 0.001

    return parameters

## Objective Function

The objective function:

- Creates a model from sampled parameters
- Trains on the training set
- Evaluates on the validation set
- Returns validation accuracy

In [24]:
def objective(trial):
    params = define_search_space(trial)

    model = DynamicCNN(
        params["number_of_conv_layers"],
        params["number_of_filters"],
        params["fully_connected_size"],
        params["dropout_probability"]
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE)

    # Training loop moved directly into the objective function
    model.train()
    for epoch in range(3):
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            predicted_values = model(x)
            loss = criterion(predicted_values, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    accuracy = MulticlassAccuracy(num_classes=NUMBER_OF_CLASSES).to(device)

    model.eval()
    with torch.no_grad():
        for x, y in validation_loader:
            x, y = x.to(device), y.to(device)
            predicted_values = model(x)
            accuracy.update(predicted_values, y)

    return accuracy.compute().item()

## Task 5: Run Optuna Study

Run a study with multiple trials to test the performance of different hyperparamter combinations.

In [ ]:
study = optuna.create_study(direction="maximize", sampler=optuna_sampler)
study.optimize(objective, n_trials=15)

[I 2026-05-21 14:23:08,429] A new study created in memory with name: no-name-bec63774-eb06-4bb0-a09b-60af371370b5
[I 2026-05-21 14:30:49,206] Trial 0 finished with value: 0.6382531523704529 and parameters: {}. Best is trial 0 with value: 0.6382531523704529.


In [ ]:
print(f"Best trial's validation accuracy: {study.best_value:.4f}")

## Task 6: Train Final Model

Now that suitable hyperparameters have been identified, train the final model on both the training and the validation data using these hyperparameters.

Since we're now training for (slightly) more epochs, add a learning-rate scheduler of your choice to reduce the learning rate during later stages of training.

<details>

<summary>Hint</summary>

Much of the required code can be re-used from the objective function.

</details>

In [ ]:
number_of_epochs = 10

best_parameters = study.best_params
print(best_parameters)

train_and_validation_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# TODO: Retrain the model using the hyperparameters that showed the best performance
model = ...

optimizer = ...
criterion = ...

# TODO: choose a suitable learning-rate scheduler (don't forget to also include it in the training loop)
scheduler = ...

for epoch in range(number_of_epochs):
    ...

## Task 7: Evaluate Final Model

Evaluate the performance of the final model on the test data.

Note that the model has never seen these data during training, nor were they used to optimize any hyperparameters.

<details>

<summary>Hint</summary>

Much of the required code can be re-used from the objective function.

</details>

In [ ]:
# TODO: compute the accuracy on the test data

test_loader = ...

accuracy = ...

...